## Pipeline

In [ ]:
import json
import math
from pathlib import Path

import numpy as np

from osgeo import gdal
from osgeo import gdal_array
from osgeo import gdalconst
from osgeo import ogr
from osgeo import osr

# ----------------------------------------------------------------------------
# clip
# ----------------------------------------------------------------------------
def clip(ll: tuple, 
         ur: tuple, 
         outSRS: osr.SpatialReference,
         ds: gdal.Dataset,
         width: int,
         height: int) -> gdal.Dataset:

    warpOptions = gdal.WarpOptions(outputBounds=[ll[0], ll[1], ur[0], ur[1]],
                                   format='MEM',
                                   outputBoundsSRS=outSRS,
                                   width=width,
                                   height=height)

    clipDs: gdal.Dataset = gdal.Warp('', ds, options=warpOptions)

    return clipDs
    
# ----------------------------------------------------------------------------
# createCube
# ----------------------------------------------------------------------------
def createCube(tileDef: dict, 
               layer: ogr.Layer,
               ll: tuple,
               ur: tuple) -> np.ndarray:

    # ---
    # We cannot know the final number of 512 x 512 images in the stack, unless
    # we read all the images beforehand; therefore, we cannot define the
    # ndarray.  Instead, store each 512 x 512 raster in a list, and use it to
    # create the ndarray at the end.
    # ---
    rasterList = []
    
    ltmSRS = osr.SpatialReference()
    ltmSRS.ImportFromWkt(tileDef['crs'])
    
    # ---
    # Read the images and put them in the cube.
    # ---
    for feature in layer:
    
        ds: gdal.Dataset = gdal.Open(feature['location'], gdalconst.GA_ReadOnly)

        # Verify data type.  Is is ndarray or Dataset?
        clipDs: np.ndarray = clip(ll, 
                                  ur, 
                                  ltmSRS, 
                                  ds, 
                                  tileDef['tileWidth'], 
                                  tileDef['tileHeight'])
    
        raster: np.ndarray = clipDs.ReadAsArray()

        # ---
        # If the raster has one band, the shape will be in two dimensions.
        # If the raster has multiple bands, the shape will be in three
        # dimensions.
        # ---
        numBands = 1 if len(raster.shape) == 2 else raster.shape[0]
        

        if numBands == 1:
            
            rasterList.append(raster)

        else:
            
            for i in range(numBands):
                rasterList.append(raster[i])

    cube = np.stack(rasterList, axis=0)
    print('shape:', cube.shape)
    
    return cube

# ----------------------------------------------------------------------------
# getTileBbox
# ----------------------------------------------------------------------------
def getTileBbox(tileDef: dict, tileIndex: tuple) -> tuple:

    tileX = tileIndex[0]
    tileY = tileIndex[1]

    gridOrigin = tileDef['pointOfOrigin']  # Upper left
    cellSize = tileDef['cellSize']
    width = tileDef['tileWidth']
    height = tileDef['tileHeight']

    ulx = gridOrigin[0] + tileX * width * cellSize
    uly = gridOrigin[1] - tileY * height * cellSize
    llx = ulx
    lly = uly - height * cellSize
    urx = ulx + width * cellSize
    ury = uly

    assert(((urx - llx) / tileDef['cellSize']) - tileDef['tileWidth'] < 1)
    assert(((ury - lly) / tileDef['cellSize']) - tileDef['tileHeight'] < 1)
    
    return ((llx, lly), (urx, ury))

# ----------------------------------------------------------------------------
# getTileDbPath
# ----------------------------------------------------------------------------
def getTileDbPath(imageDir: Path, dbName: str = 'output_index.shp') -> Path:

    if not imageDir or not imageDir.exists() or not imageDir.is_dir():
        raise ValueError('A valid image directory must be provided.')

    fullPath: Path = imageDir / dbName

    if fullPath.exists():
        return fullPath

    MOON_SRS = 'GEOGCRS["Moon (2015) - Sphere / Ocentric", DATUM["Moon (2015) - Sphere", ELLIPSOID["Moon (2015) - Sphere",1737400,0, LENGTHUNIT["metre",1]]], PRIMEM["Reference Meridian",0, ANGLEUNIT["degree",0.0174532925199433]], CS[ellipsoidal,2], AXIS["geodetic latitude (Lat)",north, ORDER[1], ANGLEUNIT["degree",0.0174532925199433]], AXIS["geodetic longitude (Lon)",east, ORDER[2], ANGLEUNIT["degree",0.0174532925199433]], ID["IAU",30100,2015], REMARK["Source of IAU Coordinate systems: https://doi.org/10.1007/s10569-017-9805-5"]]'
    outFile: Path = imageDir / dbName
    
    # The database does not exist, so create it for the image directory.
    gdal.TileIndex(outFile, list(imageDir.glob('*.tif')), outputSRS=MOON_SRS)

    return outFile
    
# ----------------------------------------------------------------------------
# getTileDef
# ----------------------------------------------------------------------------
def getTileDef(tmsDir: Path, zone: str, zoomLevel: int) -> dict:

    tmsFileName = 'tms_LTM_' + zone + 'RG.json'
    tmsPath = tmsDir / tmsFileName

    with open(tmsPath, 'r') as f:
        tms = json.load(f)

    # If we made it here, we must have the correct zone, etc.  No need to validate.
    zoomLevels: list = tms['tileMatrices']
    tileDef = [zl for zl in zoomLevels if int(zl['id']) == zoomLevel][0]

    if len(tileDef) == 0:
        
        raise RuntimeError('Unable to find zoom level ' + 
                           str(zoomLevel) + 
                           ' in ' + 
                           str(tmsPath))

    tileDef['crs'] = tms['crs']
    
    return tileDef

# ----------------------------------------------------------------------------
# latLonToLTM
# ----------------------------------------------------------------------------
def latLonToLTM(lat: float, lon: float, tileDef: dict) -> tuple[float, float]:
    
    # ---
    # Lat/lon SRS
    # ---
    LAT_LON_SRS_PROJ4 = '+proj=longlat +R=1737400 +no_defs'    
    latLonSRS = osr.SpatialReference()
    latLonSRS.ImportFromProj4(LAT_LON_SRS_PROJ4)
    
    # ---
    # LTM SRS
    # ---
    ltmSRS = osr.SpatialReference()
    ltmSRS.ImportFromWkt(tileDef['crs'])
    
    # Transform the coordinates.
    xform = osr.CoordinateTransformation(latLonSRS, ltmSRS)
    test = xform.TransformPoint(lon, lat)
    easting, northing, height = xform.TransformPoint(lon, lat)

    return easting, northing

# ----------------------------------------------------------------------------
# ltmToLatLon
# ----------------------------------------------------------------------------
def ltmToLatLon(zone: str, 
                ll: tuple, 
                ur: tuple, 
                tileDef: dict) -> tuple[float, float]:
    
    # ---
    # Lat/lon SRS
    # ---
    LAT_LON_SRS_PROJ4 = '+proj=longlat +R=1737400 +no_defs'    
    latLonSRS = osr.SpatialReference()
    latLonSRS.ImportFromProj4(LAT_LON_SRS_PROJ4)
    
    # ---
    # LTM SRS
    # ---
    ltmSRS = osr.SpatialReference()
    ltmSRS.ImportFromWkt(tileDef['crs'])
    
    # Transform the coordinates.
    xform = osr.CoordinateTransformation(ltmSRS, latLonSRS)
    llLatLon = xform.TransformPoint(ll[0], ll[1])
    urLatLon = xform.TransformPoint(ur[0], ur[1])

    return llLatLon, urLatLon

# ----------------------------------------------------------------------------
# ltmToTileIndex
# ----------------------------------------------------------------------------
def ltmToTileIndex(easting: float, 
                   northing: float, 
                   tileDef: dict) -> tuple[int, int]:

    # Extract TMS grid parameters
    originX = tileDef['pointOfOrigin'][0]
    originY = tileDef['pointOfOrigin'][1]
    cellSize = tileDef['cellSize']
    tileWidth = tileDef['tileWidth']
    tileHeight = tileDef['tileHeight']
    
    # Calculate tile size in meters
    tileSize = tileWidth * cellSize
    
    # Convert LTM coordinates to tile indices
    x = int((easting - originX) / tileSize)
    y = int((originY - northing) / tileSize)
    
    return x, y
    
# ----------------------------------------------------------------------------
# pipeline
# ----------------------------------------------------------------------------
def pipeline(repoPath: Path, 
             dbPath: Path,
             outDir: Path,
             zone: int, 
             zoomLevel: int,
             tileIndex: tuple) -> Path:

    # Read the tile definition.
    tmsDir = repoPath / 'TMS/RG'
    tileDef: dict = getTileDef(tmsDir, zone, zoomLevel)

    # Get the tile corners.
    ll, ur = getTileBbox(tileDef, tileIndex)
    
    # Query
    llLatLon, urLatLon = ltmToLatLon(zone, ll, ur, tileDef)
    layer: ogr.Layer = query(dbPath, llLatLon, urLatLon)

    cubeFile: Path = None
    
    if layer.GetFeatureCount() == 0:
        
        print('Tile does not overlap any images.')

    else:
        
        # Create the data cube.
        cube: np.ndarray = createCube(tileDef, layer, ll, ur)
        
        # ---
        # Write the data cube as a CoG.
        # ---
        cubeFile = writeCube(outDir, zone, zoomLevel, tileIndex, cube, tileDef)

    return cubeFile

# ----------------------------------------------------------------------------
# query
# ----------------------------------------------------------------------------
def query(dbPath: Path, llLatLon: tuple, urLatLon: tuple) -> ogr.Layer:

    driver: ogr.Driver = ogr.GetDriverByName('ESRI Shapefile')
    ds: ogr.Dataset = driver.Open(str(dbPath), 0)
    layer: ogr.Layer = ds.GetLayer()
    layer.SetSpatialFilterRect(llLatLon[0], llLatLon[1], urLatLon[0], urLatLon[1])

    # ---
    # When you return an ogr.Layer object from a function, the underlying 
    # DataSource that owns the layer may be getting garbage collected, making 
    # the layer invalid.  Attach the datasource to the layer to prevent 
    # garbage collection
    # ---
    layer._ds = ds
    
    return layer

# ----------------------------------------------------------------------------
# writeCube
# ----------------------------------------------------------------------------
def writeCube(outDir: Path, 
              zone: str, 
              zoomLevel: int,
              tileIndex: tuple, 
              cube: np.ndarray,
              tileDef: dict) -> Path:

    outName = 'Cube-LTM' + zone + \
              '-Zoom-' + str(zoomLevel) + \
              '-Tile-' + str(tileIndex[0]) + '-' + str(tileIndex[1]) + \
              '.tif'
    
    outFile = outDir / outName

    # ---
    # First create a temporary in-memory dataset
    # ---
    dataType = gdal_array.NumericTypeCodeToGDALTypeCode(cube.dtype)
    memDriver = gdal.GetDriverByName('MEM')
    
    memDs = memDriver.Create('', 
                             cube.shape[1],    # width
                             cube.shape[2],    # height  
                             cube.shape[0],    # bands
                             dataType)
    
    if memDs is None:
        raise RuntimeError('Failed to create dataset in memory.')
    
    ltmSRS = osr.SpatialReference()
    ltmSRS.ImportFromWkt(tileDef['crs'])
    memDs.SetSpatialRef(ltmSRS)

    geotransform = [
        tileDef['pointOfOrigin'][0],  # X-coordinate of upper-left corner
        tileDef['cellSize'],          # Pixel width (west-east resolution)
        0,                            # Rotation (usually 0)
        tileDef['pointOfOrigin'][1],  # Y-coordinate of upper-left corner  
        0,                            # Rotation (usually 0)
        -tileDef['cellSize']          # Pixel height (north-south resolution, negative!)
    ]
    
    memDs.SetGeoTransform(geotransform)

    # ---
    # Write data to temporary dataset
    # ---
    for bandIndex in range(cube.shape[0]):
        
        band = memDs.GetRasterBand(bandIndex + 1)
        band.WriteArray(cube[bandIndex])

    # ---
    # Now create COG using CreateCopy.
    # ---
    cogDriver = gdal.GetDriverByName('COG')
    
    cogDs = cogDriver.CreateCopy(str(outFile),
                                 memDs,
                                 options=['BIGTIFF=YES', 'COMPRESS=LZW'])
    
    tempDs = None
    cogDs = None

    return outFile
    

In [2]:
OUT_DIR = Path('/explore/nobackup/people/rlgill/SystemTesting/lunar/cubes')
REPO_PATH = Path('/explore/nobackup/people/rlgill/innovation-lab-repositories/Armstrong_Benchmark')

# dbPath = Path('/explore/nobackup/projects/ilab/projects/Lunar_FM/data/testdata/NAC_PHO/NAC_PHO_E186N1213/WAC/output_index.shp')
dbPath = getTileDbPath(Path('/explore/nobackup/projects/ilab/projects/Lunar_FM/data/testdata/NAC_PHO/NAC_PHO_E186N1213/WAC/'))
zone = '37S'
zoomLevel = 5

cubeFile1: Path = pipeline(REPO_PATH, dbPath, OUT_DIR, zone, zoomLevel, (0, 0))
print('cubeFile1:', cubeFile1)

cubeFile2: Path = pipeline(REPO_PATH, dbPath, OUT_DIR, zone, zoomLevel, (9, 0))
print('cubeFile2:', cubeFile2)

cubeFile3: Path = pipeline(REPO_PATH, dbPath, OUT_DIR, zone, zoomLevel, (6, 10))
print('cubeFile3:', cubeFile3)

cubeFile4: Path = pipeline(REPO_PATH, dbPath, OUT_DIR, zone, 2, (9, 0))
print('cubeFile4:', cubeFile4)


/panfs/ccds02/app/modules/jupyter/ilab/pytorch-kernel-current/lib/python3.12/site-packages/osgeo/osr.py:410: FutureWarning: Neither osr.UseExceptions() nor osr.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


Tile does not overlap any images.
cubeFile1: None
shape: (118, 512, 512)
cubeFile2: /explore/nobackup/people/rlgill/SystemTesting/lunar/cubes/Cube-LTM37S-Zoom-5-Tile-9-0.tif
shape: (32, 512, 512)
cubeFile3: /explore/nobackup/people/rlgill/SystemTesting/lunar/cubes/Cube-LTM37S-Zoom-5-Tile-6-10.tif
shape: (120, 512, 512)
cubeFile4: /explore/nobackup/people/rlgill/SystemTesting/lunar/cubes/Cube-LTM37S-Zoom-2-Tile-9-0.tif


## Include Other Image Types
- /explore/nobackup/projects/ilab/projects/Lunar_FM/data/testdata/Diviner/Hpar
- /explore/nobackup/projects/ilab/projects/Lunar_FM/data/testdata/gravity

In [3]:
dbPath = Path('/explore/nobackup/people/rlgill/SystemTesting/lunar/db.shp')
outDir = Path('/explore/nobackup/people/rlgill/SystemTesting/lunar/cubes/data2')

cubeFile5: Path = pipeline(REPO_PATH, dbPath, outDir, zone, zoomLevel, (0, 0))
print('cubeFile5:', cubeFile5)

cubeFile6: Path = pipeline(REPO_PATH, dbPath, outDir, zone, zoomLevel, (9, 0))
print('cubeFile6:', cubeFile6)


shape: (2, 512, 512)
cubeFile5: /explore/nobackup/people/rlgill/SystemTesting/lunar/cubes/data2/Cube-LTM37S-Zoom-5-Tile-0-0.tif
shape: (120, 512, 512)
cubeFile6: /explore/nobackup/people/rlgill/SystemTesting/lunar/cubes/data2/Cube-LTM37S-Zoom-5-Tile-9-0.tif


In [4]:
dbPath = Path('/explore/nobackup/people/rlgill/SystemTesting/lunar/db2.shp')
outDir = Path('/explore/nobackup/people/rlgill/SystemTesting/lunar/cubes/data3')

cubeFile7: Path = pipeline(REPO_PATH, dbPath, outDir, zone, zoomLevel, (0, 0))
print('cubeFile7:', cubeFile7)


shape: (4, 512, 512)
cubeFile7: /explore/nobackup/people/rlgill/SystemTesting/lunar/cubes/data3/Cube-LTM37S-Zoom-5-Tile-0-0.tif


## Process New Data
There is a new getTileDbPath that will create the necessary Shapefile database for the input image directory, if it does not already exist.

The area containing this data was given as 149.8 deg E/1.2 deg N.  We must convert that to a tile index.

#### To Do
- There might be extraneous coordinate conversions now that the user input is a coordinate, instead of a tile inded.
- Modify the pipeline to accept a lat/lon, instead of a tile index.

### LRO_NAC_Pho_Sites

In [19]:
# ---
# User input
# ---
imageDir = Path('/explore/nobackup/projects/lfm/processed_data/Lunar/LRO_NAC_Pho_Sites')
outDir = Path('/explore/nobackup/people/rlgill/SystemTesting/lunar/cubes/data4')
zone = '42N'
lon = 149.8
lat = 1.2

# ---
# Use the new function to retrieve the image database, creating it if necessary.
# ---
dbPath = getTileDbPath(imageDir)
print('dbPath:', dbPath)

# ---
# Find the tile index covering the point given above.  Note, this applies to the 
# zoom level defined in previous notebook cells.
# ---
tmsDir = REPO_PATH / 'TMS/RG'
tileDef: dict = getTileDef(tmsDir, zone, zoomLevel)

# Convert the input lat/lon to LTM.
easting, northing = latLonToLTM(lat, lon, tileDef)

# Get the tile index.
tileIndex = ltmToTileIndex(easting, northing, tileDef)
print('tileIndex:', tileIndex)

# ---
# Run the pipeline.
# ---
cubeFile8: Path = pipeline(REPO_PATH, dbPath, outDir, zone, zoomLevel, tileIndex)
print('cubeFile8:', cubeFile8)


dbPath: /explore/nobackup/projects/lfm/processed_data/Lunar/LRO_NAC_Pho_Sites/output_index.shp
tileIndex: (1, 63)
shape: (32, 512, 512)
cubeFile8: /explore/nobackup/people/rlgill/SystemTesting/lunar/cubes/data4/Cube-LTM42N-Zoom-5-Tile-1-63.tif


### LRO_WAC_Pho_Sites
This many files crashes GDAL when it attempts to create the output GeoTiff.

#### The following cell crashes.

In [ ]:
imageDir = Path('/explore/nobackup/projects/lfm/processed_data/Lunar/LRO_WAC_Pho_Sites')

dbPath = getTileDbPath(imageDir)
print('dbPath:', dbPath)

cubeFile9: Path = pipeline(REPO_PATH, dbPath, outDir, zone, zoomLevel, tileIndex)
print('cubeFile9:', cubeFile9)


### Mitigating the Crash
Create groups of twenty input files symbolically linked to the primary input directory.  This means we process many small batches.

This was accomplished manually, but could be automated in the pipeline.  The group size was chosen arbitrarily as a number that would definitely work.  This threshold can be changed.

In [18]:
imageDir = Path('/explore/nobackup/people/rlgill/SystemTesting/lunar/LRO_WAC_Pho_Sites')

for subdir in sorted(imageDir.iterdir()):

    if subdir.is_dir():
 
        print(f'Processing subdirectory: {subdir.name}')

        dbPath = getTileDbPath(subdir)
 
        # Make an output directory for this group.
        groupOutDir = outDir / subdir.stem
        groupOutDir.mkdir()
        
        # Note, this is using some variables set in previous notebook cells.
        cubeFile: Path = pipeline(REPO_PATH, dbPath, groupOutDir, zone, zoomLevel, tileIndex)


Processing subdirectory: group_00
shape: (70, 512, 512)
Processing subdirectory: group_01
shape: (70, 512, 512)
Processing subdirectory: group_02
shape: (70, 512, 512)
Processing subdirectory: group_03
shape: (70, 512, 512)
Processing subdirectory: group_04
shape: (70, 512, 512)
Processing subdirectory: group_05
shape: (70, 512, 512)
Processing subdirectory: group_06
shape: (70, 512, 512)
Processing subdirectory: group_07
shape: (70, 512, 512)
Processing subdirectory: group_08
shape: (70, 512, 512)
Processing subdirectory: group_09
shape: (70, 512, 512)
Processing subdirectory: group_10
shape: (70, 512, 512)
Processing subdirectory: group_11
shape: (70, 512, 512)
Processing subdirectory: group_12
shape: (70, 512, 512)
Processing subdirectory: group_13
shape: (70, 512, 512)
Processing subdirectory: group_14
shape: (70, 512, 512)
Processing subdirectory: group_15
shape: (70, 512, 512)
Processing subdirectory: group_16
shape: (70, 512, 512)
Processing subdirectory: group_17
shape: (70, 51